# 🧩 Mini-Lab: First pytest Tests, Fixtures, and Golden Files

**Module: Tests That Catch Real Bugs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** which parts of an LLM app are deterministic and can be unit tested, and which part is stochastic
2. **Write** small pytest assertions and show that each one can actually fail
3. **Use** a fixture, `tmp_path`, and a golden file to catch output regressions

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Deterministic vs Stochastic Parts | Test what your code does with outputs you control, and measure model quality separately |
| Unit Test | A small assertion about one behavior that should catch a real bug when the behavior breaks |
| Fixtures and Test Data | Reusable sample data and helpers that make tests repeatable, isolated, and easy to read |

## Setup

Create the `pytest_demo/` folder and a helper that runs pytest.

In [1]:
import subprocess
import sys
from pathlib import Path

LAB = Path("pytest_demo")
(LAB / "golden").mkdir(parents=True, exist_ok=True)

def run_pytest(*test_files):
    # Use the notebook's own Python so pytest runs the same way on Mac and Windows
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", *test_files],
        cwd=LAB, capture_output=True, text=True, encoding="utf-8",
    )
    print(result.stdout[-3000:])

## Step 1: Draw the Boundary Around the Model

> 🧩 **Deterministic vs stochastic parts.** Watch for the single comment in `call_model` that marks the stochastic boundary. Every test below stays on the deterministic side of it.

Our app classifies support tickets. Only one function talks to the model. Everything around it is plain code.

| Function | Behavior | How to check it |
|----------|----------|-----------------|
| `call_model` | Stochastic: the reply can change on every run | Measure quality separately (evals), not in unit tests |
| `build_prompt` | Deterministic pure function | Unit test |
| `parse_label` | Deterministic pure function | Unit test with model replies **you** write |
| `save_report` | Deterministic, writes a file | Unit test with a temp path + golden file |

`call_model` stays isolated. The tests never call it, so they stay fast and reliable.

In [2]:
MODULE = LAB / "ticket_tools.py"
MODULE.write_text(r'''from pathlib import Path

ALLOWED = {"billing", "bug", "feature"}

def build_prompt(ticket: str) -> str:
    return f"Classify this ticket as billing, bug, or feature.\nTicket: {ticket.strip()}\nLabel:"

def call_model(prompt: str) -> str:
    # Stochastic boundary: the real LLM call lives here. Unit tests never touch it.
    raise NotImplementedError

def parse_label(raw: str) -> str:
    label = raw.strip().strip(".").lower()
    return label if label in ALLOWED else "unknown"

def save_report(rows, path) -> None:
    lines = [f"{ticket} -> {label}" for ticket, label in rows]
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")
''', encoding="utf-8")

# Keep a copy so Step 3 can inject a bug and then restore the original
ORIGINAL_SOURCE = MODULE.read_text(encoding="utf-8")
print("Wrote", MODULE)

Wrote pytest_demo\ticket_tools.py


## Step 2: Write Your First Unit Tests

> 🧩 **Unit test.** The small, single-behavior assertion from the video, now in pytest.

Each test checks **one behavior** with a plain `assert`. The model replies (`"  Billing.\n"`, `"I think it's a bug"`) are strings we wrote by hand, so the result is always the same.

The last test, `test_theatre`, is on purpose a weak test. Keep an eye on it in Step 3.

In [3]:
(LAB / "test_parse.py").write_text('''from ticket_tools import parse_label


def test_parse_label_normalizes_case_and_punctuation():
    assert parse_label("  Billing.\\n") == "billing"


def test_parse_label_rejects_unexpected_text():
    assert parse_label("I think it's a bug") == "unknown"


def test_theatre():
    assert parse_label("Billing.") is not None
''', encoding="utf-8")

run_pytest("test_parse.py")

...                                                                      [100%]
3 passed in 0.07s



## Step 3: Prove Each Test Can Fail

A green test only means something if it turns red when the code breaks. We put in a realistic bug: someone "simplifies" `parse_label` to `raw.lower()` and drops the clean-up. Then we run the tests again and put the original code back.

Watch two things. The normalization test should catch the bug. `test_theatre` still passes, because it never checks the actual label.

In [4]:
# Inject the bug: drop the clean-up step
buggy = ORIGINAL_SOURCE.replace('raw.strip().strip(".").lower()', 'raw.lower()')
MODULE.write_text(buggy, encoding="utf-8")
print("With the bug:")
run_pytest("test_parse.py")

# Restore the original code: everything should be green again
MODULE.write_text(ORIGINAL_SOURCE, encoding="utf-8")
print("Bug reverted:")
run_pytest("test_parse.py")

With the bug:


F..                                                                      [100%]
================================== FAILURES ===================================
______________ test_parse_label_normalizes_case_and_punctuation _______________

    def test_parse_label_normalizes_case_and_punctuation():
>       assert parse_label("  Billing.\n") == "billing"
E       AssertionError: assert 'unknown' == 'billing'
E         
E         - billing
E         + unknown

test_parse.py:5: AssertionError
=========================== short test summary info ===========================
FAILED test_parse.py::test_parse_label_normalizes_case_and_punctuation - AssertionError: assert 'unknown' == 'billing'
1 failed, 2 passed in 0.19s

Bug reverted:


...                                                                      [100%]
3 passed in 0.02s



## Step 4: Fixtures, Temporary Paths, and a Golden File

> 🧩 **Fixtures and Test Data.** Reusable sample data and helpers that keep tests repeatable and isolated. Watch for `canned_replies` showing up as an argument in two tests.

- `canned_replies` is a **pytest fixture**. It holds model replies we control, and any test that names it as a parameter receives it.
- `tmp_path` is a built-in fixture. Each test gets a fresh, empty folder, so it never writes over real files.
- `golden/report.txt` is the **golden file**: the exact output we approved. If the report format ever changes, the comparison fails.

In [5]:
(LAB / "golden" / "report.txt").write_text(
    "I was charged twice -> billing\n"
    "App crashes on login -> bug\n"
    "Please add dark mode -> unknown\n",
    encoding="utf-8",
)

(LAB / "test_report.py").write_text('''from pathlib import Path

import pytest

from ticket_tools import parse_label, save_report

GOLDEN = Path(__file__).parent / "golden" / "report.txt"


@pytest.fixture
def canned_replies():
    # Model outputs we control: no real LLM call, same result every run
    return [
        ("I was charged twice", " Billing."),
        ("App crashes on login", "BUG"),
        ("Please add dark mode", "maybe a feature?"),
    ]


def test_first_canned_reply(canned_replies):
    assert parse_label(canned_replies[0][1]) == "billing"


def test_report_matches_golden(canned_replies, tmp_path):
    out = tmp_path / "report.txt"
    save_report([(t, parse_label(r)) for t, r in canned_replies], out)
    assert out.read_text(encoding="utf-8") == GOLDEN.read_text(encoding="utf-8")
''', encoding="utf-8")

run_pytest("test_parse.py", "test_report.py")

.....                                                                    [100%]
5 passed in 0.13s



## 🎯 Summary

### Key Takeaways

1. **Deterministic vs stochastic** — keep the model call behind one boundary and unit test the pure code around it with replies you control
2. **Unit tests** — one small `assert` per behavior, run with `python -m pytest`
3. **Tests must be able to fail** — inject a real bug; a test that stays green (like `test_theatre`) is theatre
4. **Fixtures and `tmp_path`** — share sample data across tests and write files to an isolated temporary folder
5. **Golden files** — compare output to an approved copy to catch format regressions